#Sitema de Gerenciamento de Tarefas/bugs
Em vez de fazer um sistema de usuários, como no exemplo disponibilizado na aula, construí um sistema que simula um gerenciador de tarefas/bugs. O código aplica os principais conceitos abordados na aula: Validação (Regex, Before, After), Tipos Especiais (EmailStr, SecretStr, UUID4), Flags Binárias (IntFlag), Serialização e a integração nativa com o FastAPI.

In [1]:
%pip install "pydantic[email]"
%pip install fastapi 

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import enum
import re
from datetime import datetime
from typing import Any, Self
from uuid import uuid4

from fastapi import FastAPI
from fastapi.responses import JSONResponse
from fastapi.testclient import TestClient
from pydantic import (
    BaseModel,
    EmailStr,
    Field,
    SecretStr,
    UUID4,
    field_serializer,
    field_validator,
    model_validator,
)

# Regex corrigido: O título deve começar com letra ou colchetes e ter pelo menos 5 caracteres. Permite letras, colchetes, pontos, espaços e acentos
VALID_TITLE_REGEX = re.compile(r"^[a-zA-Z\[\]. À-ÿ]{5,}$")

# IntFlag: Permite combinar múltiplas tags usando o operador '|'
class IssueTag(enum.IntFlag):
    BACKEND = 1
    FRONTEND = 2
    BUG = 4
    URGENT = 8

class Issue(BaseModel):
    # Proíbe que usuários enviem campos extras não mapeados (Segurança de API)
    model_config = {
        "extra": "forbid",
    }
    
    # "Banco de dados" em memória para os testes
    __issues__ = []

    id: UUID4 = Field(default_factory=uuid4, description="ID único gerado automaticamente", kw_only=True)
    title: str = Field(..., examples=["[BUG] Falha no login"])
    description: str = Field(..., description="Descrição detalhada do problema")
    reporter_email: EmailStr = Field(..., description="Email de quem reportou o erro", frozen=True)
    tags: IssueTag = Field(default=0, validate_default=True)
    
    # SecretStr: Protege dados sensíveis. exclude=True garante que não vaze no JSON.
    internal_notes: SecretStr = Field(
        default=SecretStr(""), exclude=True, description="Notas internas de segurança"
    )
    
    created_at: datetime = Field(default_factory=datetime.now, kw_only=True)
    related_issues: list[UUID4] = Field(default_factory=list, max_length=50)

    # 1. Validador de Campo Simples (Regex)
    @field_validator("title")
    @classmethod
    def validate_title(cls, v: str) -> str:
        if not VALID_TITLE_REGEX.match(v):
            raise ValueError("O título é muito curto ou possui formato inválido.")
        return v

    # 2. Validador Before (Transforma Int/Str na nossa classe IntFlag)
    @field_validator("tags", mode="before")
    @classmethod
    def parse_tags(cls, v: int | str | IssueTag) -> IssueTag:
        op = {int: lambda x: IssueTag(x), str: lambda x: IssueTag[x], IssueTag: lambda x: x}
        try:
            return op[type(v)](v)
        except (KeyError, ValueError):
            validos = ", ".join([x.name for x in IssueTag if x.name])
            raise ValueError(f"Tag inválida. Opções aceitas: {validos}")

    # 3. Validador de Modelo Before (Checa dados cruzados antes de montar o objeto)
    @model_validator(mode="before")
    @classmethod
    def validate_issue_pre(cls, v: dict[str, Any]) -> dict[str, Any]:
        if "title" in v and "description" in v:
            if v["title"].casefold() in v["description"].casefold():
                raise ValueError("A descrição não pode ser apenas uma cópia do título.")
        return v

    # 4. Validador de Modelo After (Regra de Negócio com o objeto pronto)
    @model_validator(mode="after")
    def validate_issue_post(self) -> Self:
        # Se a tag for de BUG, exige que o título indique isso claramente
        if IssueTag.BUG in self.tags and "[BUG]" not in self.title.upper():
            raise ValueError("Issues marcadas como BUG devem conter '[BUG]' no título.")
        return self

    # 5. Serializadores: Como os dados saem do sistema (JSON)
    @field_serializer("tags", when_used="json")
    def serialize_tags(self, tags: IssueTag) -> str:
        # Retorna o nome da flag em vez do número (ex: "BUG" em vez de 4)
        return tags.name if tags.name else str(tags.value)

    @field_serializer("id", when_used="json")
    def serialize_id(self, id: UUID4) -> str:
        return str(id)


# INTEGRAÇÃO COM FASTAPI 
app = FastAPI()

@app.get("/issues", response_model=list[Issue])
async def get_issues() -> list[Issue]:
    return list(Issue.__issues__)

@app.post("/issues", response_model=Issue)
async def create_issue(issue: Issue):
    Issue.__issues__.append(issue)
    return issue

@app.get("/issues/{issue_id}", response_model=Issue)
async def get_issue(issue_id: UUID4) -> Issue | JSONResponse:
    try:
        return next((i for i in Issue.__issues__ if i.id == issue_id))
    except StopIteration:
        return JSONResponse(status_code=404, content={"message": "Issue não encontrada"})


# TESTES AUTOMATIZADOS
def main() -> None:
    with TestClient(app) as client:
        # Teste 1: Criar uma Issue com sucesso (Combinando Tags FRONTEND + BUG)
        response = client.post(
            "/issues",
            json={
                "title": "[BUG] Botão não funciona",
                "description": "Ao clicar no botão de salvar, a página trava completamente.",
                "reporter_email": "dev@empresa.com",
                "tags": 6, # 2 (FRONTEND) + 4 (BUG) = 6
                "internal_notes": "Possível vazamento de memória"
            },
        )
        assert response.status_code == 200
        data = response.json()
        assert "internal_notes" not in data, "Segurança falhou: notas internas vazaram!"
        print("Teste 1 Passou: Issue criada com sucesso e dados sensíveis ocultados.")

        # Teste 2: Falha na Regra de Negócio (@model_validator mode="after")
        response = client.post(
            "/issues",
            json={
                "title": "Erro no sistema",
                "description": "Tela azul ao iniciar.",
                "reporter_email": "dev@empresa.com",
                "tags": "BUG", # Pydantic tentará processar como BUG
            },
        )
        assert response.status_code == 422 # Erro de Validação
        assert "devem conter '[BUG]' no título" in response.text
        print("Teste 2 Passou: Pydantic barrou Bug sem a nomenclatura correta.")

        # Teste 3: Falha no Validador Before (Descrição igual ao Título)
        response = client.post(
            "/issues",
            json={
                "title": "[BUG] Tela preta",
                "description": "[BUG] tela preta",
                "reporter_email": "teste@email.com",
            },
        )
        assert response.status_code == 422
        assert "não pode ser apenas uma cópia" in response.text
        print("Teste 3 Passou: Validação de modelo impediu descrição preguiçosa.")

        # Teste 4: Segurança de API (Enviando campo extra)
        response = client.post(
            "/issues",
            json={
                "title": "[BUG] Banco de dados lento",
                "description": "Queries demorando mais de 5s.",
                "reporter_email": "dba@empresa.com",
                "campo_hacker": "tentando injetar dados"
            },
        )
        assert response.status_code == 422
        print("Teste 4 Passou: FastAPI barrou atributos extras (extra='forbid').\n")
        
        print("Todos os testes passaram! API está robusta e segura.")


if __name__ == "__main__":
    main()

Teste 1 Passou: Issue criada com sucesso e dados sensíveis ocultados.
Teste 2 Passou: Pydantic barrou Bug sem a nomenclatura correta.
Teste 3 Passou: Validação de modelo impediu descrição preguiçosa.
Teste 4 Passou: FastAPI barrou atributos extras (extra='forbid').

Todos os testes passaram! API está robusta e segura.
